# Aula 01: Setup, Importação e Diagnóstico Estrutural
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

Nesta primeira aula prática, vamos:
1. Importar a biblioteca `pandas` e carregar a planilha bruta `Campe_Supply_Corrompido.xlsx`.
2. Fazer o diagnóstico estrutural da base (`shape`, `info`, `head`, `isna`).
3. Padronizar os nomes das colunas (boas práticas de *snake_case*).
4. Descobrir e resolver a **armadilha das colunas inúteis** que escondem linhas duplicadas.

In [ ]:
import pandas as pd
import numpy as np

# Configurando o pandas para mostrar todas as colunas na tela
pd.set_option('display.max_columns', None)

## 1. Carregando e Inspecionando a Base de Dados
Sempre que recebemos um arquivo Excel, o primeiro passo é verificar quais abas existem dentro dele e inspecionar as primeiras linhas.

In [ ]:
# Verificando o nome das abas do arquivo Excel
caminho_arquivo = 'Campe_Supply_Corrompido.xlsx'
xls = pd.ExcelFile(caminho_arquivo)
print('Abas disponíveis na planilha:', xls.sheet_names)

# Lendo a aba principal
df = pd.read_excel(caminho_arquivo, sheet_name='Campe Supply')
df.head(10)

## 2. Diagnóstico Inicial (Tamanho, Tipos e Nulos)
Vamos usar `.shape`, `.info()` e `.isna().sum()` para responder:
* Quantas linhas e colunas temos?
* As colunas numéricas (`FATURAMENTO`, `margem de lucro`, `LUCRO`) estão realmente como números (`float`) ou como texto (`object`)?
* Quantos valores vazios existem em cada coluna?

In [ ]:
print(f'Formato da base: {df.shape[0]} linhas e {df.shape[1]} colunas\n')
df.info()

In [ ]:
# Contando a quantidade e o percentual de valores nulos por coluna
resumo_nulos = pd.DataFrame({
    'Total_Nulos': df.isna().sum(),
    'Percentual (%)': (df.isna().mean() * 100).round(2)
})
resumo_nulos

## 3. A Armadilha das Colunas Inúteis e das Duplicatas
Repare nas colunas `UNNAMED: 0`, `coluna_inutil_0` e `coluna_inutil_1`. Vamos investigar o que há dentro delas e testar se a base possui linhas duplicadas antes e depois de removê-las.

In [ ]:
# Conferindo os valores que aparecem em UNNAMED: 0
print('Valores em UNNAMED: 0:')
print(df['UNNAMED: 0'].value_counts(dropna=False).head(10))

# Testando duplicatas ANTES de remover as colunas inúteis
print('\nDuplicatas ANTES de remover colunas inúteis:', df.duplicated().sum())

In [ ]:
# Removendo as 3 colunas que só contêm lixo/códigos aleatórios
colunas_lixo = ['UNNAMED: 0', 'coluna_inutil_0', 'coluna_inutil_1']
df = df.drop(columns=colunas_lixo)

# Testando duplicatas DEPOIS de remover as colunas inúteis!
print('Duplicatas DEPOIS de remover colunas inúteis:', df.duplicated().sum())

**Por que isso aconteceu?**
As colunas `coluna_inutil_0` e `coluna_inutil_1` tinham códigos aleatórios diferentes em cada linha. Enquanto elas estavam na tabela, o Python achava que nenhuma linha era igual à outra! Ao removê-las, descobrimos **35 vendas exatamente duplicadas**.

In [ ]:
# Removendo as 35 linhas duplicadas e reorganizando o índice
df = df.drop_duplicates().reset_index(drop=True)
print(f'Novo formato da base após remover duplicatas: {df.shape}')

## 4. Padronização dos Nomes das Colunas
Na tabela original, temos colunas em maiúsculo (`SETOR`), minúsculo (`região`) e com espaços/acentos (`data da venda`, `margem de lucro`). Vamos padronizar tudo para **snake_case** (minúsculo, sem acento e separado por `_`).

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('ã', 'a')
)

print('Colunas padronizadas:', df.columns.tolist())
df.head()

## 5. Salvando o Checkpoint da Aula 01
Para encerrar a Aula 01, vamos exportar o nosso progresso ou deixar o código pronto para a Aula 02.

In [ ]:
# Opcional: salvar o checkpoint da Aula 01
# df.to_excel('checkpoint_aula01.xlsx', index=False)
print('Aula 01 concluída com sucesso!')